# PlantSeg — Round 3, Notebook K: verified artifacts, frozen-pipeline evaluation

**One notebook, one run, every number traceable to a SHA-256.** This notebook answers the
Round-3 reviewer requests that need a GPU:

| Section | What it produces | Reviewer point |
|---|---|---|
| 1 | Re-hashes every released checkpoint and refuses to run if any hash differs from `MANIFEST.csv` | R4-3, R5-1 |
| 2 | Evaluates **every** released classifier (val + test, original view, flipped view, flip-TTA) and compares each test prediction with the per-image record saved at training time | R4-3, R4-4, R5-1 |
| 3 | Evaluates every released segmenter (val + test), severity agreement | R4-3, R4-4 |
| 4 | **Pre-declared, validation-only model selection** (rule printed before any test number is used) | R5-2, R4-1 |
| 5 | Runs the frozen decision-support pipeline once on test, on the de-duplicated clean test set, and on validation | R4-4, R5-3 |
| 6 | Does the confidence flag add anything beyond softmax confidence? | R5-4 |
| 7 | Attribution checks with the exact checkpoints: classifier null models with paired effect sizes + CIs; all segmenters | R4-2, R4-4 |
| 8 | Checkpoint-to-result map, environment record, zip of all outputs | R4-3, R4-4 |

**Before running:** attach (1) the PlantSeg dataset (the one with `Metadata.csv`,
`images/`, `annotation_{train,val,test}.json`) and (2) the `Checkpoint_Release` dataset
(the folder containing `MANIFEST.csv`). Use a **T4** GPU (not P100). Internet must be ON
(SegFormer-B2's config is fetched from the Hugging Face hub). Then *Run All*.
Expected run time on a T4: roughly 1.5–2.5 hours. Download
`/kaggle/working/plantseg_r3_notebookK_results.zip` at the end.

In [1]:
# ===================== SECTION 0: environment =====================
import importlib, os, subprocess, sys, time

def hb(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

hb("Notebook K starting")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for pkg, mod in [("segmentation-models-pytorch", "segmentation_models_pytorch"),
                 ("grad-cam", "pytorch_grad_cam"), ("timm", "timm"),
                 ("albumentations", "albumentations"), ("scikit-image", "skimage"),
                 ("statsmodels", "statsmodels")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        hb(f"installing {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# SegFormer-B2's checkpoint loads strictly only under transformers==4.46.3 (see Notebook I);
# huggingface_hub must be pinned with it, and the Xet backend disabled (see Notebook I/J).
TRANSFORMERS_PIN, HF_HUB_PIN = "4.46.3", "0.26.2"
for m in list(sys.modules):
    if m.split(".")[0] in ("transformers", "huggingface_hub", "hf_xet"):
        del sys.modules[m]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"huggingface_hub=={HF_HUB_PIN}", f"transformers=={TRANSFORMERS_PIN}"], check=True)

import json, glob, hashlib, random, warnings, math, zipfile, platform, copy, shutil, gc
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import timm, segmentation_models_pytorch as smp
import albumentations as A
import transformers as _tf, huggingface_hub as _hfh
from transformers import SegformerForSemanticSegmentation
from pytorch_grad_cam import GradCAM
from scipy import stats
from skimage.filters import threshold_otsu
_trapz = getattr(np, "trapezoid", None) or np.trapz
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, cohen_kappa_score)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import StratifiedTable
warnings.filterwarnings("ignore")

assert _tf.__version__ == TRANSFORMERS_PIN, f"transformers {_tf.__version__} != {TRANSFORMERS_PIN}"
assert _hfh.__version__ == HF_HUB_PIN, f"huggingface_hub {_hfh.__version__} != {HF_HUB_PIN}"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = bool(int(os.environ.get("PLANTSEG_SMOKE", "0")))   # local smoke test only; leave 0 on Kaggle

ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "timm": timm.__version__,
       "segmentation_models_pytorch": smp.__version__, "albumentations": A.__version__,
       "transformers": _tf.__version__, "huggingface_hub": _hfh.__version__,
       "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
       "cudnn_deterministic": True, "run_started": time.strftime("%Y-%m-%d %H:%M:%S")}
hb(f"environment: {ENV}")

[16:29:48] Notebook K starting
[16:29:48] installing segmentation-models-pytorch
[16:29:51] installing grad-cam


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.26.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.26.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.26.2 which is incompatible.
I0000 00:00:1795019459.337790      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


[16:31:07] environment: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'timm': '1.0.30', 'segmentation_models_pytorch': '0.6.0', 'albumentations': '2.0.8', 'transformers': '4.46.3', 'huggingface_hub': '0.26.2', 'gpu': 'Tesla T4', 'cudnn_deterministic': True, 'run_started': '2026-11-18 16:31:07'}


In [2]:
# ===================== SECTION 1: locate inputs, verify every checkpoint hash =====================
def find_data_root():
    env = os.environ.get("PLANTSEG_DATA")
    if env and (Path(env)/"Metadata.csv").exists(): return Path(env)
    for c in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"), Path("/kaggle/input/plantseg")]:
        if (c/"Metadata.csv").exists(): return c
    hits = [h for h in glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)]
    assert hits, "Metadata.csv not found -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

def find_release_root():
    env = os.environ.get("PLANTSEG_RELEASE")
    if env and (Path(env)/"MANIFEST.csv").exists(): return Path(env)
    hits = glob.glob("/kaggle/input/**/MANIFEST.csv", recursive=True)
    assert hits, "MANIFEST.csv not found -- attach the Checkpoint_Release dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root(); RELEASE = find_release_root()
OUT = Path(os.environ.get("PLANTSEG_OUT", "/kaggle/working/notebookK_results")); OUT.mkdir(parents=True, exist_ok=True)
hb(f"DATA_ROOT={DATA_ROOT}  RELEASE={RELEASE}  OUT={OUT}")

MAN = pd.read_csv(RELEASE/"MANIFEST.csv")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""): h.update(chunk)
    return h.hexdigest()

hash_rows = []
for r in MAN.itertuples():
    p = RELEASE/r.file
    got = sha256_file(p) if p.exists() else "MISSING"
    hash_rows.append({"file": r.file, "sha256_manifest": r.sha256, "sha256_this_run": got,
                      "match": got == r.sha256})
hash_df = pd.DataFrame(hash_rows); hash_df.to_csv(OUT/"K1_hash_verification.csv", index=False)
hb(f"hash check: {hash_df.match.sum()}/{len(hash_df)} files match MANIFEST.csv")
assert hash_df.match.all(), "A checkpoint differs from MANIFEST.csv -- STOP: do not report numbers from it.\n" + \
    hash_df[~hash_df.match].to_string()
SHA = dict(zip(MAN.file, MAN.sha256))

def find_previous(name):
    """Return an output file from an earlier run of this notebook (in OUT, or attached as an
    input via Add Input -> Your Work), so finished sections are not recomputed."""
    for c in [OUT/name] + [Path(p) for p in sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))]:
        if c.exists(): return c
    return None

def reuse_previous(table_name, per_file_prefix, task):
    prev = find_previous(table_name)
    if prev is None: return None
    stems = [Path(f).stem for f in MAN[MAN.task == task].file]
    if not all((prev.parent/f"{per_file_prefix}{s}.csv").exists() for s in stems): return None
    tab = pd.read_csv(prev)
    if set(tab.sha256) != set(MAN[MAN.task == task].sha256): return None   # different checkpoints -> recompute
    if prev.parent != OUT:
        for p in prev.parent.glob(f"{table_name.split('_')[0]}_*.csv"): shutil.copy(p, OUT/p.name)
    hb(f"reusing completed {task} results from an earlier run: {prev.parent}")
    return tab

[16:31:15] DATA_ROOT=/kaggle/input/datasets/alenmaruf/plantseg/plantseg  RELEASE=/kaggle/input/datasets/alenmaruf/checkpoint-release/Checkpoint_Release  OUT=/kaggle/working/notebookK_results
[16:32:18] hash check: 43/43 files match MANIFEST.csv


In [3]:
# ===================== SECTION 1b: data index, masks, image caches =====================
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training": "train", "Validation": "val", "Test": "test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
CLASS_NAMES = sorted(meta["Disease"].unique().tolist()); NUM_CLASSES = len(CLASS_NAMES)
L2I = {c: i for i, c in enumerate(CLASS_NAMES)}
assert NUM_CLASSES == 115
val_df = meta[meta.split == "val"].reset_index(drop=True)
test_df = meta[meta.split == "test"].reset_index(drop=True)
assert len(val_df) == 846 and len(test_df) == 1561, (len(val_df), len(test_df))

flags = pd.read_csv(RELEASE/"splits/test_group_nonindependent_flags_311.csv")
test_df["leak_flag"] = test_df["Name"].map(dict(zip(flags.image, flags.is_flagged))).astype(bool)
assert int(test_df.leak_flag.sum()) == 311
val_df["leak_flag"] = False
SUB300 = pd.read_csv(RELEASE/"splits/attribution_subset_300.csv")["image"].tolist()
assert len(SUB300) == 300

if SMOKE:   # tiny local run: a few images per split, keeping the leak flag and subset populated
    keep_t = list(test_df[test_df.leak_flag].index[:6]) + list(test_df[~test_df.leak_flag].index[:10])
    keep_t += list(test_df[test_df.Name.isin(SUB300)].index[:6])
    test_df = test_df.loc[sorted(set(keep_t))].reset_index(drop=True); val_df = val_df.head(16).copy()
    SUB300 = [n for n in SUB300 if n in set(test_df.Name)]

def load_anns(split):
    d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn = {}
    for im in d["images"]:
        polys = [s for a in by.get(im["id"], []) for s in a["segmentation"] if len(s) >= 6]
        fn[im["file_name"]] = (im["width"], im["height"], polys)
    return fn

def rasterize(w, h, polys):
    m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
    for s in polys: dr.polygon(list(zip(s[0::2], s[1::2])), fill=1)
    return m

# Classifier preprocessing = the Stage-1 training notebooks' eval transform
# (torchvision Resize on the PIL image). Segmenter preprocessing = the Stage-2 training
# notebooks' eval transform (albumentations Resize: bilinear image, nearest mask).
_seg_resize = A.Resize(512, 512)
SUB300_SET = set(SUB300)
CACHE, MASK512, GT_SEV, NATIVE_MASK = {}, {}, {}, {}
for split, df in [("val", val_df), ("test", test_df)]:
    t0 = time.time(); anns = load_anns(split); n = len(df)
    X224 = np.zeros((n, 224, 224, 3), np.uint8); X384 = np.zeros((n, 384, 384, 3), np.uint8)
    X512 = np.zeros((n, 512, 512, 3), np.uint8); M = np.zeros((n, 512, 512), bool); sev = np.zeros(n)
    for i, r in enumerate(df.itertuples()):
        img = Image.open(r.filepath).convert("RGB")
        X224[i] = np.asarray(T.Resize((224, 224))(img)); X384[i] = np.asarray(T.Resize((384, 384))(img))
        w, h, polys = anns.get(r.Name, (None, None, []))
        if w is None: w, h = img.size
        m = np.asarray(rasterize(w, h, polys))
        out = _seg_resize(image=np.asarray(img), mask=m)
        X512[i] = out["image"]; M[i] = out["mask"] > 0; sev[i] = float((m > 0).mean())
        if split == "test" and r.Name in SUB300_SET: NATIVE_MASK[r.Name] = m > 0
    CACHE[(split, 224)], CACHE[(split, 384)], CACHE[(split, 512)] = X224, X384, X512
    MASK512[split] = M; GT_SEV[split] = sev
    hb(f"cached {split}: {n} images in {time.time()-t0:.0f}s")
mr = test_df["Mask ratio"].astype(float).values
hb(f"GT severity (rasterised polygons) vs Metadata 'Mask ratio': r={np.corrcoef(GT_SEV['test'], mr)[0,1]:.4f}, "
   f"max |diff|={np.abs(GT_SEV['test']-mr).max():.4f}")
Y = {"val": val_df["Disease"].map(L2I).values, "test": test_df["Disease"].map(L2I).values}
CLEAN = ~test_df["leak_flag"].values

_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
def to_x(u8):
    x = torch.from_numpy(np.ascontiguousarray(u8)).to(device).permute(0, 3, 1, 2).float() / 255.0
    return (x - _MEAN) / _STD

[16:34:19] cached val: 846 images in 120s
[16:37:40] cached test: 1561 images in 201s
[16:37:40] GT severity (rasterised polygons) vs Metadata 'Mask ratio': r=0.9943, max |diff|=0.5020


In [4]:
# ===================== SECTION 1c: model builders (verbatim from the training notebooks) =====================
IMG_SIZE = {"ConvNeXtV2Tiny": 384, "EfficientNetV2S": 384, "ResNet50": 384, "MobileNetV3Large": 384,
            "EfficientNetB0": 384, "ViTBase16": 224, "SwinTiny": 224, "GatedFusionLight": 224}

def build_classifier(name, nc=NUM_CLASSES):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, nc); return m
    if name == "EfficientNetB0": return timm.create_model("efficientnet_b0", pretrained=False, num_classes=nc)
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, nc); return m
    if name == "EfficientNetV2S": return timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=nc)
    if name == "ConvNeXtV2Tiny": return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=nc)
    if name == "ViTBase16": return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=nc)
    if name == "SwinTiny": return timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=nc)
    if name == "GatedFusionLight":
        class _Gated(nn.Module):
            def __init__(self, ba, da, bb, db, nc, gh=128, hh=256, dr=0.3):
                super().__init__(); self.branch_a = ba; self.branch_b = bb; fused = da + db
                self.gate = nn.Sequential(nn.Linear(fused, gh), nn.ReLU(), nn.Linear(gh, 2), nn.Softmax(dim=1))
                self.classifier = nn.Sequential(nn.Linear(fused, hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh, nc))
            def forward(self, x):
                fa, fb = self.branch_a(x), self.branch_b(x); g = self.gate(torch.cat([fa, fb], 1))
                return self.classifier(torch.cat([fa*g[:, 0:1], fb*g[:, 1:2]], 1))
        mb = torchvision.models.mobilenet_v3_large(weights=None)
        da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
        b = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, global_pool="avg")
        return _Gated(mb, da, b, b.num_features, nc)
    raise ValueError(name)

SMP_SPECS = {"UNet_resnet34": (smp.Unet, "resnet34"), "UNetPP_resnet50": (smp.UnetPlusPlus, "resnet50"),
             "DeepLabV3Plus_resnet50": (smp.DeepLabV3Plus, "resnet50"),
             "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
             "FPN_resnet50": (smp.FPN, "resnet50")}

class SegNet(nn.Module):
    def __init__(self, name, nc=2):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            self.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); self.kind = "smp"
        else:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                "nvidia/mit-b2", num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
    def forward(self, x):
        if self.kind == "smp": return self.net(x)
        lg = self.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

def load_ckpt(model, path):
    raw = torch.load(path, map_location="cpu")
    if isinstance(raw, dict):
        for k in ("state_dict", "model_state_dict", "model", "best_state"):
            if k in raw and isinstance(raw[k], dict): raw = raw[k]; break
    raw = {k.replace("module.", ""): v for k, v in raw.items()}
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if len(missing) >= 0.5*len(model.state_dict()):
        raise RuntimeError(f"{path}: {len(missing)} keys missing -- refusing to evaluate random weights")
    return model, len(missing), len(unexpected)

def arch_seed(stem):
    import re
    m = re.match(r"(.+?)_seed(\d+)", stem); return m.group(1), int(m.group(2))

In [5]:
# ===================== SECTION 2: every released classifier, val + test, TTA diagnostics =====================
def cls_metrics(y, p):
    return {"acc": accuracy_score(y, p), "bal_acc": balanced_accuracy_score(y, p),
            "macro_f1": f1_score(y, p, average="macro", zero_division=0), "mcc": matthews_corrcoef(y, p)}

@torch.no_grad()
def cls_logits(model, X, bs=48):
    lo, lf = [], []
    for i in range(0, len(X), bs):
        x = to_x(X[i:i+bs]); lo.append(model(x).float().cpu()); lf.append(model(torch.flip(x, dims=[3])).float().cpu())
    return torch.cat(lo), torch.cat(lf)

def historical_file(stem):
    arch, seed = arch_seed(stem)
    if stem == "ConvNeXtV2Tiny_seed42_retrained": return "ConvNeXtV2Tiny_seed42retrained_test_predictions_historical.csv"
    return f"{arch}_seed{seed}_test_predictions_historical.csv"   # superseded file is compared with the 78.4% record

CLS_MAN = MAN[MAN.task == "classifier"].reset_index(drop=True)
cls_eval = reuse_previous("K2_classifier_checkpoint_eval.csv", "K2_test_predictions_", "classifier")
if cls_eval is None:
    cls_rows, PRED = [], {}
    for r in CLS_MAN.itertuples():
        stem = Path(r.file).stem; arch, seed = arch_seed(stem); t0 = time.time()
        model, nmiss, nunexp = load_ckpt(build_classifier(arch), RELEASE/r.file); model = model.to(device).eval()
        s = IMG_SIZE[arch]; row = {"file": r.file, "sha256": r.sha256, "architecture": arch, "seed": seed,
                                   "status": r.status, "missing_keys": nmiss, "unexpected_keys": nunexp, "img_size": s}
        for split in ("val", "test"):
            lo, lf = cls_logits(model, CACHE[(split, s)]); y = Y[split]
            po, pf, pt = lo.argmax(1).numpy(), lf.argmax(1).numpy(), (lo+lf).argmax(1).numpy()
            conf = F.softmax(lo, 1).max(1).values.numpy()
            PRED[(stem, split)] = {"orig": po, "flip": pf, "tta": pt, "conf_single": conf}
            for tag, p in (("single", po), ("tta", pt)):
                for k, v in cls_metrics(y, p).items(): row[f"{split}_{k}_{tag}"] = v
            if split == "test":
                row["test_acc_flip"] = float((pf == y).mean())
                row["test_view_agreement"] = float((po == pf).mean())
                row["test_n_corrected_by_tta"] = int(((po != y) & (pt == y)).sum())
                row["test_n_harmed_by_tta"] = int(((po == y) & (pt != y)).sum())
                row["test_n_unchanged_by_tta"] = int(len(y) - row["test_n_corrected_by_tta"] - row["test_n_harmed_by_tta"])
                for tag, p in (("single", po), ("tta", pt)):
                    row[f"clean_acc_{tag}"] = float((p == y)[CLEAN].mean())
                    row[f"flagged_acc_{tag}"] = float((p == y)[~CLEAN].mean())
                    row[f"clean_macro_f1_{tag}"] = f1_score(y[CLEAN], p[CLEAN], average="macro", zero_division=0)
                hf = RELEASE/"historical_records"/historical_file(stem)
                if hf.exists():
                    h = pd.read_csv(hf).set_index("image").loc[test_df.Name]
                    row["historical_acc"] = float((h.y_true.values == h.y_pred.values).mean())
                    row["historical_agreement_tta"] = float((h.y_pred.values == pt).mean())
                    row["historical_agreement_single"] = float((h.y_pred.values == po).mean())
                    row["reproduces_historical"] = bool(row["historical_agreement_tta"] >= 0.98 and
                                                        abs(row["test_acc_tta"] - row["historical_acc"]) <= 0.005)
                pd.DataFrame({"image": test_df.Name, "y_true": y, "pred_single": po, "pred_flip": pf, "pred_tta": pt,
                              "softmax_conf_single": conf, "leak_flag": test_df.leak_flag}).to_csv(
                    OUT/f"K2_test_predictions_{stem}.csv", index=False)
        cls_rows.append(row)
        hb(f"{stem:45s} val_acc={row['val_acc_single']:.4f} test_tta={row['test_acc_tta']:.4f} "
           f"single={row['test_acc_single']:.4f} hist={row.get('historical_acc', float('nan')):.4f} "
           f"agree={row.get('historical_agreement_tta', float('nan')):.4f} ({time.time()-t0:.0f}s)")
        del model; torch.cuda.empty_cache()
    cls_eval = pd.DataFrame(cls_rows); cls_eval.to_csv(OUT/"K2_classifier_checkpoint_eval.csv", index=False)
print(cls_eval[["file", "val_acc_single", "test_acc_single", "test_acc_flip", "test_acc_tta", "test_view_agreement",
                "test_n_corrected_by_tta", "test_n_harmed_by_tta", "clean_acc_tta", "historical_acc",
                "historical_agreement_tta", "reproduces_historical"]].round(4).to_string(index=False))

[16:37:48] reusing completed classifier results from an earlier run: /kaggle/input/datasets/alenmaruf/result-sec-1-4/notebookK_results
                                                                  file  val_acc_single  test_acc_single  test_acc_flip  test_acc_tta  test_view_agreement  test_n_corrected_by_tta  test_n_harmed_by_tta  clean_acc_tta  historical_acc  historical_agreement_tta  reproduces_historical
                         classifiers/seed123/ConvNeXtV2Tiny_seed123.pt          0.7470           0.7258         0.7245        0.7277               0.9154                       25                    22         0.7272          0.7277                    1.0000                   True
                         classifiers/seed123/EfficientNetB0_seed123.pt          0.7175           0.7380         0.7373        0.7341               0.9321                       14                    20         0.7376          0.7341                    1.0000                   True
                      

In [6]:
# ===================== SECTION 3: every released segmenter, val + test, severity =====================
def ccc(a, b):
    a, b = np.asarray(a), np.asarray(b); ma, mb = a.mean(), b.mean()
    return 2*np.mean((a-ma)*(b-mb)) / (a.var() + b.var() + (ma-mb)**2)

def bands(s): return np.digitize(s, [0.01, 0.10, 0.25])

def severity_stats(pred, gt):
    d = pred - gt
    return {"ccc": ccc(gt, pred), "pearson": stats.pearsonr(gt, pred)[0], "spearman": stats.spearmanr(gt, pred)[0],
            "mae": float(np.abs(d).mean()), "rmse": float(np.sqrt((d**2).mean())), "ba_bias": float(d.mean()),
            "ba_loa": float(1.96*d.std()), "kappa_linear": cohen_kappa_score(bands(gt), bands(pred), weights="linear")}

@torch.no_grad()
def seg_predict(model, split, bs=6):
    X, M = CACHE[(split, 512)], MASK512[split]
    tp = fp = fn = tn = 0; per_iou, psev = [], []
    for i in range(0, len(X), bs):
        pr = (model(to_x(X[i:i+bs])).argmax(1) > 0).cpu().numpy(); gt = M[i:i+bs]
        tp += int((pr & gt).sum()); fp += int((pr & ~gt).sum()); fn += int((~pr & gt).sum()); tn += int((~pr & ~gt).sum())
        for a, b in zip(pr, gt):
            u = (a | b).sum(); per_iou.append((a & b).sum()/u if u > 0 else 1.0); psev.append(a.mean())
    fg = tp/(tp+fp+fn); bg = tn/(tn+fp+fn)
    return {"fg_iou": fg, "fg_dice": 2*tp/(2*tp+fp+fn), "miou": (fg+bg)/2, "pixel_acc": (tp+tn)/(tp+fp+fn+tn)}, \
           np.array(per_iou), np.array(psev)

SEG_MAN = MAN[MAN.task == "segmenter"].reset_index(drop=True)
seg_eval = reuse_previous("K3_segmenter_checkpoint_eval.csv", "K3_test_per_image_", "segmenter")
if seg_eval is None:
    seg_rows, SEGPRED = [], {}
    for r in SEG_MAN.itertuples():
        stem = Path(r.file).stem; arch, seed = arch_seed(stem); t0 = time.time()
        model, nmiss, nunexp = load_ckpt(SegNet(arch), RELEASE/r.file); model = model.to(device).eval()
        row = {"file": r.file, "sha256": r.sha256, "architecture": arch, "seed": seed, "missing_keys": nmiss, "unexpected_keys": nunexp}
        for split in ("val", "test"):
            pooled, pi, ps = seg_predict(model, split, bs=3 if "UNetPP" in arch else 6)
            for k, v in pooled.items(): row[f"{split}_{k}"] = v
            row[f"{split}_perimage_fg_iou"] = float(pi.mean()); SEGPRED[(stem, split)] = {"iou": pi, "sev": ps}
            if split == "test":
                for name, msk in (("clean", CLEAN), ("flagged", ~CLEAN)):
                    row[f"{name}_perimage_fg_iou"] = float(pi[msk].mean())
                for name, msk in (("test", np.ones(len(pi), bool)), ("clean", CLEAN)):
                    for k, v in severity_stats(ps[msk], GT_SEV["test"][msk]).items(): row[f"{name}_sev_{k}"] = v
                hf = RELEASE/"historical_records"/f"{arch}_seed{seed}_test_per_image_fgiou_historical.csv"
                if hf.exists():
                    h = pd.read_csv(hf).set_index("image").loc[test_df.Name]["fg_iou"].values
                    row["historical_perimage_fg_iou"] = float(h.mean())
                    row["historical_max_abs_diff"] = float(np.abs(h - pi).max())
                    row["reproduces_historical"] = bool(np.abs(h - pi).mean() < 1e-3)
                pd.DataFrame({"image": test_df.Name, "fg_iou": pi, "pred_severity": ps, "gt_severity": GT_SEV["test"],
                              "leak_flag": test_df.leak_flag}).to_csv(OUT/f"K3_test_per_image_{stem}.csv", index=False)
        seg_rows.append(row)
        hb(f"{stem:45s} val_fg={row['val_fg_iou']:.4f} test_fg={row['test_fg_iou']:.4f} "
           f"sevCCC={row['test_sev_ccc']:.4f} reproduces={row.get('reproduces_historical')} ({time.time()-t0:.0f}s)")
        del model; torch.cuda.empty_cache()
    seg_eval = pd.DataFrame(seg_rows); seg_eval.to_csv(OUT/"K3_segmenter_checkpoint_eval.csv", index=False)
print(seg_eval[["file", "val_fg_iou", "test_fg_iou", "clean_perimage_fg_iou", "test_sev_ccc", "clean_sev_ccc",
                "historical_perimage_fg_iou", "reproduces_historical"]].round(4).to_string(index=False))

[16:37:54] reusing completed segmenter results from an earlier run: /kaggle/input/datasets/alenmaruf/result-sec-1-4/notebookK_results
                                                       file  val_fg_iou  test_fg_iou  clean_perimage_fg_iou  test_sev_ccc  clean_sev_ccc  historical_perimage_fg_iou  reproduces_historical
segmenters/seed123/DeepLabV3Plus_efficientnet-b3_seed123.pt      0.6440       0.6498                 0.5854        0.8098         0.7979                      0.5927                   True
       segmenters/seed123/DeepLabV3Plus_resnet50_seed123.pt      0.5744       0.5786                 0.5072        0.7620         0.7514                      0.5129                   True
                 segmenters/seed123/FPN_resnet50_seed123.pt      0.6001       0.6120                 0.5395        0.7919         0.7678                      0.5483                   True
                 segmenters/seed123/SegFormer_b2_seed123.pt      0.6144       0.6346                 0.5610       

In [7]:
# ===================== SECTION 4: PRE-DECLARED, VALIDATION-ONLY MODEL SELECTION =====================
RULE = """
PRE-DECLARED SELECTION RULE (fixed before any test-set number is used; uses validation data only)
Classifier:
  * Candidates: the five single-backbone CNN classifiers (ConvNeXtV2Tiny, EfficientNetV2S, ResNet50,
    MobileNetV3Large, EfficientNetB0). The confidence flag needs a Grad-CAM map whose lesion
    localisation has been checked against ground truth; transformers (ViT-B/16, Swin-Tiny) and the
    two-branch GatedFusionLight have no single convolutional target layer and are therefore not
    eligible for the pipeline. They are still reported in the benchmark.
  * Eligible checkpoints: released (not superseded), loads with 0 missing keys.
  * Architecture score: mean single-pass validation accuracy over its eligible seeds (42, 123, 7).
  * Deployed checkpoint: the selected architecture's eligible checkpoint with the highest
    single-pass validation accuracy.
Segmenter:
  * Candidates: all six segmenters; eligible = released, 0 missing keys.
  * Architecture score: mean pooled validation foreground IoU over seeds; deployed checkpoint =
    that architecture's seed with the highest pooled validation foreground IoU.
No threshold, temperature or calibration parameter is fitted anywhere in the pipeline: the
segmenter mask is argmax, and the confidence flag is the parameter-free pointing-game test.
The test set is used once, afterwards, to evaluate the frozen pipeline.
"""
print(RULE)
CNN_CANDIDATES = ["ConvNeXtV2Tiny", "EfficientNetV2S", "ResNet50", "MobileNetV3Large", "EfficientNetB0"]
elig = cls_eval[(cls_eval.status == "release") & (cls_eval.missing_keys == 0)]
cls_score = elig.groupby("architecture")["val_acc_single"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
cand_score = cls_score.loc[[a for a in cls_score.index if a in CNN_CANDIDATES]]
SEL_CLS_ARCH = cand_score.index[0]
sel_cls_row = elig[elig.architecture == SEL_CLS_ARCH].sort_values("val_acc_single", ascending=False).iloc[0]
SEL_CLS_FILE = sel_cls_row.file
elig_s = seg_eval[seg_eval.missing_keys == 0]
seg_score = elig_s.groupby("architecture")["val_fg_iou"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
SEL_SEG_ARCH = seg_score.index[0]
SEL_SEG_FILE = elig_s[elig_s.architecture == SEL_SEG_ARCH].sort_values("val_fg_iou", ascending=False).iloc[0].file
selection = {"rule": RULE, "classifier_validation_ranking_all": cls_score.round(4).reset_index().to_dict("records"),
             "classifier_validation_ranking_candidates": cand_score.round(4).reset_index().to_dict("records"),
             "unconstrained_validation_winner": cls_score.index[0],
             "selected_classifier_architecture": SEL_CLS_ARCH, "selected_classifier_file": SEL_CLS_FILE,
             "selected_classifier_sha256": SHA[SEL_CLS_FILE],
             "segmenter_validation_ranking": seg_score.round(4).reset_index().to_dict("records"),
             "selected_segmenter_architecture": SEL_SEG_ARCH, "selected_segmenter_file": SEL_SEG_FILE,
             "selected_segmenter_sha256": SHA[SEL_SEG_FILE]}
json.dump(selection, open(OUT/"K4_selection.json", "w"), indent=2, default=str)
print(cls_score.round(4).to_string()); print(seg_score.round(4).to_string())
hb(f"SELECTED classifier: {SEL_CLS_FILE}   segmenter: {SEL_SEG_FILE}")


PRE-DECLARED SELECTION RULE (fixed before any test-set number is used; uses validation data only)
Classifier:
  * Candidates: the five single-backbone CNN classifiers (ConvNeXtV2Tiny, EfficientNetV2S, ResNet50,
    MobileNetV3Large, EfficientNetB0). The confidence flag needs a Grad-CAM map whose lesion
    localisation has been checked against ground truth; transformers (ViT-B/16, Swin-Tiny) and the
    two-branch GatedFusionLight have no single convolutional target layer and are therefore not
    eligible for the pipeline. They are still reported in the benchmark.
  * Eligible checkpoints: released (not superseded), loads with 0 missing keys.
  * Architecture score: mean single-pass validation accuracy over its eligible seeds (42, 123, 7).
  * Deployed checkpoint: the selected architecture's eligible checkpoint with the highest
    single-pass validation accuracy.
Segmenter:
  * Candidates: all six segmenters; eligible = released, 0 missing keys.
  * Architecture score: mean pooled v

In [8]:
# ===================== SECTION 5: the frozen pipeline, run once on test (and on val, clean test) =====================
gc.collect(); torch.cuda.empty_cache()
PIPE_BS = 4   # Grad-CAM keeps the full backward graph; 16 images at 384 px exhausted a 16 GB T4

def target_layer(arch, m):
    return {"ConvNeXtV2Tiny": lambda: m.stages[-1], "EfficientNetV2S": lambda: m.bn2, "EfficientNetB0": lambda: m.bn2,
            "ResNet50": lambda: m.layer4[-1], "MobileNetV3Large": lambda: m.features[-1]}[arch]()

clf, _, _ = load_ckpt(build_classifier(SEL_CLS_ARCH), RELEASE/SEL_CLS_FILE); clf = clf.to(device).eval()
seg, _, _ = load_ckpt(SegNet(SEL_SEG_ARCH), RELEASE/SEL_SEG_FILE); seg = seg.to(device).eval()
cam_extractor = GradCAM(model=clf, target_layers=[target_layer(SEL_CLS_ARCH, clf)])
S_CLS = IMG_SIZE[SEL_CLS_ARCH]

def flag_from(cam_cls, pred_mask):
    """Stage-5 flag, unchanged: CAM resized to the mask grid, binarised at the predicted
    severity for the IoU statistic; the flag itself is the parameter-free pointing game."""
    cam_rs = np.asarray(Image.fromarray((cam_cls*255).astype(np.uint8)).resize((512, 512))).astype(np.float32)/255.0
    sevp = float(pred_mask.mean()); thr = np.quantile(cam_rs, 1 - max(sevp, 1e-3))
    cb = cam_rs >= thr; u = (cb | pred_mask).sum()
    iou = float((cb & pred_mask).sum()/u) if u > 0 else 0.0
    py, px = np.unravel_index(np.argmax(cam_rs), cam_rs.shape)
    return iou, bool(pred_mask[py, px]), cam_rs

def run_pipeline(split, df):
    rows = []; Xc, Xs = CACHE[(split, S_CLS)], CACHE[(split, 512)]
    for i in range(0, len(df), PIPE_BS):
        xc = to_x(Xc[i:i+PIPE_BS])
        with torch.no_grad():
            lg = clf(xc); lgf = clf(torch.flip(xc, dims=[3])); pr = F.softmax(lg, 1)
            masks = (seg(to_x(Xs[i:i+PIPE_BS])).argmax(1) > 0).cpu().numpy()
        cams = cam_extractor(input_tensor=xc, targets=None)
        for j in range(len(xc)):
            k = i + j; pred = int(pr[j].argmax()); iou, hit, _ = flag_from(cams[j], masks[j])
            rows.append({"image": df.Name.iloc[k], "y_true": int(Y[split][k]), "pred": pred,
                         "pred_tta": int((lg[j]+lgf[j]).argmax()), "correct": pred == int(Y[split][k]),
                         "softmax_conf": float(pr[j, pred]), "flag": "HIGH" if hit else "LOW",
                         "cam_pred_iou": iou, "pred_severity": float(masks[j].mean()),
                         "gt_severity": float(GT_SEV[split][k]), "leak_flag": bool(df.leak_flag.iloc[k])})
    out = pd.DataFrame(rows); out["severity_abs_error"] = (out.pred_severity - out.gt_severity).abs(); return out

def wald_gap(a1, n1, a0, n0):
    d = a1 - a0; se = math.sqrt(a1*(1-a1)/n1 + a0*(1-a0)/n0); return d, d - 1.96*se, d + 1.96*se

def summarise_pipeline(d, label):
    hi, lo = d[d.flag == "HIGH"], d[d.flag == "LOW"]
    tab = [[int(hi.correct.sum()), int((~hi.correct).sum())], [int(lo.correct.sum()), int((~lo.correct).sum())]]
    orr, pf = stats.fisher_exact(tab); gap = wald_gap(hi.correct.mean(), len(hi), lo.correct.mean(), len(lo))
    return {"set": label, "n": len(d), "accuracy_single": d.correct.mean(), "accuracy_tta": (d.pred_tta == d.y_true).mean(),
            "high_rate": len(hi)/len(d), "n_high": len(hi), "n_low": len(lo),
            "acc_high": hi.correct.mean(), "acc_low": lo.correct.mean(), "gap": gap[0], "gap_ci_lo": gap[1], "gap_ci_hi": gap[2],
            "fisher_p": pf, "table_high_correct": tab[0][0], "table_high_wrong": tab[0][1],
            "table_low_correct": tab[1][0], "table_low_wrong": tab[1][1],
            "mean_cam_pred_iou": d.cam_pred_iou.mean(), "mean_pred_severity": d.pred_severity.mean(),
            "mean_gt_severity": d.gt_severity.mean(), "sev_mae_high": hi.severity_abs_error.mean(),
            "sev_mae_low": lo.severity_abs_error.mean(),
            "sev_mwu_p": stats.mannwhitneyu(hi.severity_abs_error, lo.severity_abs_error).pvalue if len(hi) and len(lo) else np.nan,
            "spearman_camiou_sev_err": stats.spearmanr(d.cam_pred_iou, d.severity_abs_error)[0],
            "spearman_camiou_sev_err_p": stats.spearmanr(d.cam_pred_iou, d.severity_abs_error)[1],
            "severity_ccc": ccc(d.gt_severity, d.pred_severity)}

pipe_test = run_pipeline("test", test_df); pipe_val = run_pipeline("val", val_df)
_chk = cls_eval.set_index("file").loc[SEL_CLS_FILE, "test_acc_single"]
hb(f"consistency check: pipeline single-pass accuracy {pipe_test.correct.mean():.4f} vs Section 2 {_chk:.4f} "
   f"(should agree to within ~0.001; tiny differences can come from batch size on the GPU)")
pipe_test.to_csv(OUT/"K5_pipeline_per_image_test.csv", index=False); pipe_val.to_csv(OUT/"K5_pipeline_per_image_val.csv", index=False)
pipe_sum = pd.DataFrame([summarise_pipeline(pipe_test, "test_full"),
                         summarise_pipeline(pipe_test[~pipe_test.leak_flag], "test_clean_1250"),
                         summarise_pipeline(pipe_test[pipe_test.leak_flag], "test_flagged_311"),
                         summarise_pipeline(pipe_val, "validation_model_selection_evidence")])
pipe_sum.to_csv(OUT/"K5_pipeline_summary.csv", index=False)
print(pipe_sum.round(4).T.to_string())

[16:43:29] consistency check: pipeline single-pass accuracy 0.7687 vs Section 2 0.7687 (should agree to within ~0.001; tiny differences can come from batch size on the GPU)
                                   0                1                 2                                    3
set                        test_full  test_clean_1250  test_flagged_311  validation_model_selection_evidence
n                               1561             1250               311                                  846
accuracy_single               0.7687            0.776            0.7395                               0.7813
accuracy_tta                  0.7687           0.7768            0.7363                               0.7813
high_rate                     0.4042            0.384            0.4855                               0.3936
n_high                           631              480               151                                  333
n_low                            930              770           

In [9]:
# ===================== SECTION 6: does the flag add anything beyond softmax confidence? =====================
def flag_value(d, label, n_boot=2000):
    y = d.correct.astype(int).values; s = np.clip(d.softmax_conf.values, 1e-6, 1-1e-6)
    f = (d.flag == "HIGH").astype(int).values; c = d.cam_pred_iou.values; ls = np.log(s/(1-s)); n = len(y)
    res = {"set": label, "n": n, "auroc_softmax": roc_auc_score(y, s), "auroc_flag": roc_auc_score(y, f),
           "auroc_cam_pred_iou": roc_auc_score(y, c)}
    m0 = sm.Logit(y, sm.add_constant(ls)).fit(disp=0)
    m1 = sm.Logit(y, sm.add_constant(np.column_stack([ls, f]))).fit(disp=0)
    m2 = sm.Logit(y, sm.add_constant(np.column_stack([ls, f, c]))).fit(disp=0)
    res.update({"lrt_flag_given_softmax_chi2": 2*(m1.llf-m0.llf), "lrt_flag_given_softmax_p": stats.chi2.sf(2*(m1.llf-m0.llf), 1),
                "flag_or_given_softmax": float(np.exp(m1.params[2])),
                "flag_or_ci_lo": float(np.exp(m1.conf_int()[2][0])), "flag_or_ci_hi": float(np.exp(m1.conf_int()[2][1])),
                "lrt_flag_and_camiou_given_softmax_chi2": 2*(m2.llf-m0.llf),
                "lrt_flag_and_camiou_given_softmax_p": stats.chi2.sf(2*(m2.llf-m0.llf), 2)})
    Xc = np.column_stack([ls, f, c]); cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    p_comb = cross_val_predict(LogisticRegression(max_iter=2000), Xc, y, cv=cv, method="predict_proba")[:, 1]
    p_soft = cross_val_predict(LogisticRegression(max_iter=2000), ls.reshape(-1, 1), y, cv=cv, method="predict_proba")[:, 1]
    res["auroc_softmax_cv"] = roc_auc_score(y, p_soft); res["auroc_combined_cv"] = roc_auc_score(y, p_comb)
    rng = np.random.RandomState(SEED); deltas = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y[idx])) < 2: continue
        deltas.append(roc_auc_score(y[idx], p_comb[idx]) - roc_auc_score(y[idx], p_soft[idx]))
    res["delta_auroc_cv"] = res["auroc_combined_cv"] - res["auroc_softmax_cv"]
    res["delta_auroc_ci_lo"], res["delta_auroc_ci_hi"] = np.percentile(deltas, [2.5, 97.5])
    q = pd.qcut(s, 5, labels=False, duplicates="drop"); tables, bins = [], []
    for b in sorted(np.unique(q)):
        k = q == b; t = np.array([[np.sum(f[k] & y[k]), np.sum(f[k] & (1-y[k]))], [np.sum((1-f[k]) & y[k]), np.sum((1-f[k]) & (1-y[k]))]])
        bins.append({"softmax_quintile": int(b), "n": int(k.sum()), "acc_high": float(y[k][f[k] == 1].mean()) if f[k].sum() else np.nan,
                     "acc_low": float(y[k][f[k] == 0].mean()) if (1-f[k]).sum() else np.nan})
        if (t.sum(1) > 0).all() and (t.sum(0) > 0).all(): tables.append(t + 0.0)
    st = StratifiedTable(tables); cmh = st.test_null_odds(correction=False)
    res.update({"cmh_pooled_or": st.oddsratio_pooled, "cmh_or_ci_lo": st.oddsratio_pooled_confint()[0],
                "cmh_or_ci_hi": st.oddsratio_pooled_confint()[1], "cmh_p": cmh.pvalue})
    for cov in (0.5, 0.7, 0.9):
        kk = int(round(cov*n))
        res[f"selective_acc_softmax_cov{int(cov*100)}"] = y[np.argsort(-p_soft)[:kk]].mean()
        res[f"selective_acc_combined_cov{int(cov*100)}"] = y[np.argsort(-p_comb)[:kk]].mean()
    return res, pd.DataFrame(bins).assign(set=label)

fv, fb = zip(*[flag_value(pipe_test, "test_full"), flag_value(pipe_test[~pipe_test.leak_flag], "test_clean_1250"),
               flag_value(pipe_val, "validation")])
flag_val = pd.DataFrame(list(fv)); flag_val.to_csv(OUT/"K6_flag_added_value.csv", index=False)
pd.concat(fb).to_csv(OUT/"K6_flag_by_softmax_quintile.csv", index=False)
print(flag_val.round(4).T.to_string())

                                                0                1           2
set                                     test_full  test_clean_1250  validation
n                                            1561             1250         846
auroc_softmax                              0.7931           0.8232      0.7515
auroc_flag                                 0.5143           0.5104      0.5167
auroc_cam_pred_iou                         0.5114           0.5133      0.4909
lrt_flag_given_softmax_chi2                 0.365           0.0266      0.1382
lrt_flag_given_softmax_p                   0.5457           0.8704      0.7101
flag_or_given_softmax                      1.0865           1.0266      1.0717
flag_or_ci_lo                              0.8299           0.7489      0.7437
flag_or_ci_hi                              1.4225           1.4073      1.5444
lrt_flag_and_camiou_given_softmax_chi2     0.3937           0.0272      0.1383
lrt_flag_and_camiou_given_softmax_p        0.8213   

In [10]:
# ===================== SECTION 7a: classifier attribution vs ground truth, null models, effect sizes =====================
def cam_gt_metrics(cam, gt):
    if gt.sum() == 0: return np.nan, np.nan
    thr = np.quantile(cam, 1 - gt.mean()); cb = cam >= thr
    u = (cb | gt).sum(); py, px = np.unravel_index(np.argmax(cam), cam.shape)
    return float((cb & gt).sum()/u) if u > 0 else np.nan, float(gt[py, px])

def centre_prior_map(h, w, sigma_frac=0.2):
    yy, xx = np.mgrid[0:h, 0:w]; cy, cx = (h-1)/2.0, (w-1)/2.0; s = sigma_frac*min(h, w)
    g = np.exp(-(((yy-cy)**2 + (xx-cx)**2)/(2*s**2))); return g/(g.max()+1e-8)

def resize_map(cam, shape_hw):
    return np.asarray(Image.fromarray((cam*255).astype(np.uint8)).resize((shape_hw[1], shape_hw[0]))).astype(np.float32)/255.0

@torch.no_grad()
def deletion_auc_cls(model, x, cam, cls_idx, steps=11):
    order = np.argsort(-cam.ravel()); n = cam.size; xs = []
    mean = x.mean(dim=(2, 3), keepdim=True)
    for f in np.linspace(0, 1, steps):
        k = int(round(f*n)); m = np.zeros(n, bool); m[order[:k]] = True
        mt = torch.from_numpy(m.reshape(cam.shape)).to(x.device)[None, None]
        xs.append(torch.where(mt, mean, x))
    p = F.softmax(model(torch.cat(xs)), 1)[:, cls_idx].cpu().numpy()
    return float(_trapz(p, np.linspace(0, 1, steps)))

def paired_effect(a, b, n_boot=2000):
    a, b = np.asarray(a, float), np.asarray(b, float); k = ~(np.isnan(a) | np.isnan(b)); a, b = a[k], b[k]; d = a - b
    rng = np.random.RandomState(SEED); boots = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(n_boot)]
    nz = d[d != 0]; rk = stats.rankdata(np.abs(nz)); wp, wm = rk[nz > 0].sum(), rk[nz < 0].sum()
    try: p = stats.wilcoxon(a, b).pvalue
    except Exception: p = np.nan
    return {"n": int(len(d)), "mean_a": a.mean(), "mean_b": b.mean(), "mean_diff": d.mean(),
            "diff_ci_lo": np.percentile(boots, 2.5), "diff_ci_hi": np.percentile(boots, 97.5),
            "cohen_dz": d.mean()/d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan,
            "rank_biserial": (wp - wm)/(wp + wm) if (wp + wm) > 0 else 0.0, "wilcoxon_p": p}

def mcnemar_p(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); k = ~(np.isnan(a) | np.isnan(b)); a, b = a[k] > 0.5, b[k] > 0.5
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    return stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 > 0 else 1.0

sub_idx = [int(np.where(test_df.Name.values == nm)[0][0]) for nm in SUB300]
clf_rand = build_classifier(SEL_CLS_ARCH).to(device).eval()      # same architecture, fresh random weights
cam_rand = GradCAM(model=clf_rand, target_layers=[target_layer(SEL_CLS_ARCH, clf_rand)])
RNG = np.random.RandomState(SEED); attr_rows = []
for k in sub_idx:
    nm = test_df.Name.iloc[k]; gt = NATIVE_MASK[nm]; x = to_x(CACHE[("test", S_CLS)][k:k+1])
    with torch.no_grad(): pred = int(clf(x).argmax(1))
    cam = cam_extractor(input_tensor=x, targets=None)[0]; camr = cam_rand(input_tensor=x, targets=None)[0]
    real = cam_gt_metrics(resize_map(cam, gt.shape), gt); rnd_init = cam_gt_metrics(resize_map(camr, gt.shape), gt)
    rnd = cam_gt_metrics(RNG.rand(*gt.shape), gt); cen = cam_gt_metrics(centre_prior_map(*gt.shape), gt)
    attr_rows.append({"image": nm, "correct": pred == int(Y["test"][k]), "lesion_frac": float(gt.mean()),
                      "real_iou": real[0], "real_hit": real[1], "randinit_iou": rnd_init[0], "randinit_hit": rnd_init[1],
                      "random_iou": rnd[0], "random_hit": rnd[1], "centre_iou": cen[0], "centre_hit": cen[1],
                      "deletion_auc": deletion_auc_cls(clf, x, cam, pred)})
attr = pd.DataFrame(attr_rows); attr.to_csv(OUT/"K7_classifier_attribution_per_image.csv", index=False)
eff = []
for base in ("centre", "random", "randinit"):
    for metric in ("iou", "hit"):
        e = paired_effect(attr[f"real_{metric}"], attr[f"{base}_{metric}"])
        if metric == "hit": e["mcnemar_p"] = mcnemar_p(attr["real_hit"], attr[f"{base}_hit"])
        eff.append({"comparison": f"classifier real vs {base}", "metric": metric, **e})
cc, ic = attr[attr.correct], attr[~attr.correct]
eff.append({"comparison": "classifier correct vs incorrect", "metric": "iou", "n": len(attr), "mean_a": cc.real_iou.mean(),
            "mean_b": ic.real_iou.mean(), "mean_diff": cc.real_iou.mean() - ic.real_iou.mean(),
            "wilcoxon_p": stats.mannwhitneyu(cc.real_iou.dropna(), ic.real_iou.dropna()).pvalue})
cls_attr_eff = pd.DataFrame(eff); cls_attr_eff.to_csv(OUT/"K7_classifier_attribution_effects.csv", index=False)
print(attr[["real_iou", "real_hit", "deletion_auc", "randinit_iou", "random_iou", "centre_iou"]].mean().round(4).to_string())
print(cls_attr_eff.round(5).to_string(index=False))
del clf_rand, cam_rand; torch.cuda.empty_cache()

real_iou        0.1993
real_hit        0.3733
deletion_auc    0.3444
randinit_iou    0.1554
random_iou      0.1358
centre_iou      0.2804
                     comparison metric   n  mean_a  mean_b  mean_diff  diff_ci_lo  diff_ci_hi  cohen_dz  rank_biserial  wilcoxon_p  mcnemar_p
      classifier real vs centre    iou 300 0.19931 0.28039   -0.08108    -0.10179    -0.05949  -0.41620       -0.45386     0.00000        NaN
      classifier real vs centre    hit 300 0.37333 0.46667   -0.09333    -0.16667    -0.02000  -0.14080       -0.20896     0.01557    0.01934
      classifier real vs random    iou 300 0.19931 0.13581    0.06351     0.04738     0.07966   0.44959        0.45280     0.00000        NaN
      classifier real vs random    hit 300 0.37333 0.19000    0.18333     0.12000     0.25000   0.32539        0.52381     0.00000    0.00000
    classifier real vs randinit    iou 300 0.19931 0.15536    0.04395     0.02440     0.06382   0.26106        0.29678     0.00002        NaN
    classi

In [11]:
# ===================== SECTION 7b: segmenter attribution for every released segmenter =====================
class _Hook:
    def __init__(self, module):
        self.a = self.g = None
        self.fh = module.register_forward_hook(lambda m, i, o: setattr(self, "a", o))
        self.bh = module.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0]))
    def remove(self): self.fh.remove(); self.bh.remove()

def gradcam_seg(model, x):
    h = _Hook(model.net.decoder); model.zero_grad(set_to_none=True)
    lg = model(x); lg[:, 1].sum().backward()
    a, g = h.a[0].detach(), h.g[0].detach(); h.remove()
    cam = torch.relu((g.mean(dim=(1, 2))[:, None, None]*a).sum(0)); cam = cam/(cam.max()+1e-8)
    return F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].cpu().numpy()

def rollout_seg(model, x):
    with torch.no_grad(): out = model.net.segformer(pixel_values=x, output_attentions=True)
    stages, cur, cur_n = [], [], None
    for a in out.attentions:
        n = a.shape[-2]
        if cur_n is None or n == cur_n: cur.append(a); cur_n = n
        else: stages.append(cur); cur, cur_n = [a], n
    stages.append(cur); roll = None
    for a in stages[-1]:
        Am = a[0].mean(0); Am = Am/Am.sum(-1, keepdim=True).clamp_min(1e-8)
        Ah = 0.5*Am + 0.5*torch.eye(Am.shape[0], device=Am.device); roll = Ah if roll is None else Ah @ roll
    imp = roll.mean(0); side = int(round(imp.shape[0]**0.5))
    grid = imp[:side*side].reshape(side, side).float(); grid = grid/(grid.max()+1e-8)
    return F.interpolate(grid[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].cpu().numpy()

@torch.no_grad()
def deletion_auc_seg(model, x, cam, steps=11):
    order = np.argsort(-cam.ravel()); n = cam.size; mean = x.mean(dim=(2, 3), keepdim=True); ps = []
    for f in np.linspace(0, 1, steps):
        k = int(round(f*n)); m = np.zeros(n, bool); m[order[:k]] = True
        mt = torch.from_numpy(m.reshape(cam.shape)).to(x.device)[None, None]
        ps.append(float(F.softmax(model(torch.where(mt, mean, x)), 1)[:, 1].mean()))
    return float(_trapz(ps, np.linspace(0, 1, steps)))

def thr_iou(cam, gt, thr):
    cb = cam >= thr; u = (cb | gt).sum(); return float((cb & gt).sum()/u) if u > 0 else np.nan

seg_attr_rows = []
for r in SEG_MAN.itertuples():
    stem = Path(r.file).stem; arch, seed = arch_seed(stem); t0 = time.time()
    model, _, _ = load_ckpt(SegNet(arch), RELEASE/r.file); model = model.to(device).eval()
    for k in sub_idx:
        gt = MASK512["test"][k]
        if gt.sum() == 0: continue
        x = to_x(CACHE[("test", 512)][k:k+1])
        cam = rollout_seg(model, x) if arch == "SegFormer_b2" else gradcam_seg(model, x)
        iou, hit = cam_gt_metrics(cam, gt); rnd = cam_gt_metrics(RNG.rand(*gt.shape), gt)
        cen = cam_gt_metrics(centre_prior_map(*gt.shape), gt)
        try: otsu = thr_iou(cam, gt, threshold_otsu(cam))
        except Exception: otsu = np.nan
        seg_attr_rows.append({"file": r.file, "architecture": arch, "seed": seed, "image": test_df.Name.iloc[k],
                              "lesion_frac": float(gt.mean()), "real_iou": iou, "real_hit": hit,
                              "random_iou": rnd[0], "random_hit": rnd[1], "centre_iou": cen[0], "centre_hit": cen[1],
                              "median_thr_iou": thr_iou(cam, gt, np.median(cam)), "otsu_iou": otsu,
                              "deletion_auc": deletion_auc_seg(model, x, cam)})
    hb(f"attribution {stem} done ({time.time()-t0:.0f}s)")
    del model; torch.cuda.empty_cache()
seg_attr = pd.DataFrame(seg_attr_rows); seg_attr.to_csv(OUT/"K7_segmenter_attribution_per_image.csv", index=False)
seg_attr["size_bin"] = pd.cut(seg_attr.lesion_frac, [0, 0.05, 0.15, 0.30, 1.0], labels=["tiny", "small", "medium", "large"], include_lowest=True)
agg = seg_attr.groupby("file")[["real_iou", "real_hit", "deletion_auc", "median_thr_iou", "otsu_iou", "random_iou", "centre_iou"]].mean()
rows = []
for f, g in seg_attr.groupby("file"):
    row = {"file": f}
    for base in ("random", "centre"):
        for metric in ("iou", "hit"):
            e = paired_effect(g[f"real_{metric}"], g[f"{base}_{metric}"], n_boot=1000)
            row[f"{metric}_vs_{base}_diff"] = e["mean_diff"]; row[f"{metric}_vs_{base}_ci_lo"] = e["diff_ci_lo"]
            row[f"{metric}_vs_{base}_ci_hi"] = e["diff_ci_hi"]; row[f"{metric}_vs_{base}_p"] = e["wilcoxon_p"]
            row[f"{metric}_vs_{base}_rank_biserial"] = e["rank_biserial"]
    rows.append(row)
seg_attr_sum = agg.join(pd.DataFrame(rows).set_index("file")); seg_attr_sum.to_csv(OUT/"K7_segmenter_attribution_summary.csv")
seg_attr.groupby(["file", "size_bin"])[["real_iou", "real_hit"]].mean().to_csv(OUT/"K7_segmenter_attribution_by_lesion_size.csv")
print(seg_attr_sum[["real_iou", "real_hit", "deletion_auc", "otsu_iou", "median_thr_iou"]].round(4).to_string())

[16:48:39] attribution DeepLabV3Plus_efficientnet-b3_seed123 done (168s)
[17:08:23] attribution DeepLabV3Plus_resnet50_seed123 done (1185s)
[17:11:19] attribution FPN_resnet50_seed123 done (176s)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[17:14:58] attribution SegFormer_b2_seed123 done (219s)
[17:32:11] attribution UNetPP_resnet50_seed123 done (1033s)
[17:35:34] attribution UNet_resnet34_seed123 done (203s)
[17:38:19] attribution DeepLabV3Plus_efficientnet-b3_seed42 done (165s)
[17:57:56] attribution DeepLabV3Plus_resnet50_seed42 done (1177s)
[18:00:52] attribution FPN_resnet50_seed42 done (175s)


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[18:04:26] attribution SegFormer_b2_seed42 done (214s)
[18:21:40] attribution UNetPP_resnet50_seed42 done (1034s)
[18:25:02] attribution UNet_resnet34_seed42 done (203s)
[18:27:48] attribution DeepLabV3Plus_efficientnet-b3_seed7 done (165s)
[18:47:24] attribution DeepLabV3Plus_resnet50_seed7 done (1176s)
[18:50:20] attribution FPN_resnet50_seed7 done (176s)


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[18:53:55] attribution SegFormer_b2_seed7 done (215s)
[19:11:08] attribution UNetPP_resnet50_seed7 done (1033s)
[19:14:31] attribution UNet_resnet34_seed7 done (203s)
                                                             real_iou  real_hit  deletion_auc  otsu_iou  median_thr_iou
file                                                                                                                   
segmenters/seed123/DeepLabV3Plus_efficientnet-b3_seed123.pt    0.5010    0.9033        0.1278    0.5055          0.2130
segmenters/seed123/DeepLabV3Plus_resnet50_seed123.pt           0.4445    0.8333        0.1163    0.4208          0.2140
segmenters/seed123/FPN_resnet50_seed123.pt                     0.4675    0.9000        0.0834    0.4469          0.2155
segmenters/seed123/SegFormer_b2_seed123.pt                     0.2417    0.0667        0.1532    0.0526          0.2022
segmenters/seed123/UNetPP_resnet50_seed123.pt                  0.3930    0.8933        0.0722    0.4466          

In [12]:
# ===================== SECTION 8: checkpoint-to-result map, environment, zip =====================
cmap = []
for r in MAN.itertuples():
    uses = []
    if r.task == "classifier":
        uses.append("classification benchmark: val/test/clean-test metrics and TTA diagnostics (K2)")
        if r.status == "superseded": uses = ["diagnostic only: shows the Round-2 file does not reproduce the 78.4% record (K2)"]
        elif Path(r.file).stem.split("_seed")[0] in CNN_CANDIDATES: uses.append("validation-only model selection (K4)")
        if r.file == SEL_CLS_FILE:
            uses += ["deployed classifier: decision-support pipeline, confidence flag, flag added-value (K5, K6)",
                     "classifier attribution vs ground truth and null models (K7a)"]
    else:
        uses += ["segmentation benchmark and severity agreement (K3)", "validation-only model selection (K4)",
                 "segmenter attribution vs ground truth, null models, thresholds, lesion size (K7b)"]
        if r.file == SEL_SEG_FILE: uses.append("deployed segmenter: decision-support pipeline (K5, K6)")
    cmap.append({"file": r.file, "sha256": r.sha256, "used_for": " | ".join(uses)})
pd.DataFrame(cmap).to_csv(OUT/"K8_checkpoint_to_result_map.csv", index=False)
ENV["run_finished"] = time.strftime("%Y-%m-%d %H:%M:%S"); ENV["smoke"] = SMOKE
json.dump(ENV, open(OUT/"K8_environment.json", "w"), indent=2)
zp = OUT.parent/"plantseg_r3_notebookK_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")): z.write(p, p.name)
hb(f"DONE. Download {zp} ({zp.stat().st_size/1e6:.1f} MB) and bring it back for the paper and letter.")

[19:14:34] DONE. Download /kaggle/working/plantseg_r3_notebookK_results.zip (1.9 MB) and bring it back for the paper and letter.
